# One score over every subpopulation, and over every group

The Population Fidelity Score is a first attempt at collapsing three separate
population readings into one number: how close a model lands on each
subpopulation, how widely it spreads across them, and whether that spread runs
along the survey's own social divides. It arrived as a scalar over the whole
run. This notebook takes it apart and asks it again inside every group of
subpopulations the survey actually distinguishes.

`PFS = geometric_mean(accuracy, dispersion, structure)` over

| term | from | reads |
|---|---|---|
| `accuracy` | `1 - E`, `E = mean_i nEMD(WVS_i, LLM_i)` | 1 when the model lands on every cell |
| `dispersion` | `min(AR, 1/AR)`, `AR = D_LLM / D_WVS` | 1 when the model spreads exactly as far as the survey; penalised symmetrically for compression and for over-spread |
| `structure` | `max(0, rho)`, Spearman over the two condensed pairwise-nEMD vectors | 1 when the model's cell-to-cell distances rank like the survey's |

A geometric mean is a bottleneck, not an average: one term at zero puts the
score at zero however good the other two are. That is the intended reading, and
`structure` is almost always the term that binds. The component plates exist so
that every zero can be traced to the term that caused it, and every row of every
table carries `binding_term`, the name of its smallest of the three.

**One thing to raise with the supervisor before this metric is used to rank
anything.** `max(0, rho)` is discontinuous at zero, and the score is a product,
so a model that correlates at `rho = -0.0001` scores exactly zero while one at
`rho = +0.0001` keeps its other two terms. That is not a hypothetical here: 19
of the 116 pooled rows land at zero, and every one of them does so on a `rho`
between `-0.026` and `-0.0001` — values no one would call negative structure,
only absent structure. The `all` view then multiplies four topics together, so
a single such topic — nearly always religion — zeroes a model's whole row, and
16 of the 29 pooled `all` rows are zero for that reason. Nothing here smooths
it: `pfs` is computed exactly as specified, the zeros are drawn as zeros, and
the ranking plate names the binding term beside every zero bar so the cliff is
visible instead of silent. If the intent was "no credit for absent structure"
rather than "no score at all", a floor on the structure term or an arithmetic
mean of the three would both keep the ordering the components plates show.

**Center is reported beside PFS, never inside it.** `C = nEMD(mean_i WVS_i,
mean_i LLM_i)` and its score `1 - C` answer a different question — did the
model relocate its average opinion toward the target culture — and a model can
buy that cheaply while staying flat across cells. Both readings are in the
table: `pfs` is the three-term score the plates lead with, `pfs_with_center`
is the four-term form for anyone who wants the centre folded in. The plates
carry center as its own axis so the trade is visible rather than averaged away.

## Which numbers this reads

Recomputed from the run distributions, not from the derived CSVs — the same
path `population_adaptability.ipynb` takes. Two ties hold that recomputation to
the split per-run outputs `combined-metric-inputs.csv` was aggregated from:

- every per-cell `nEMD` here equals the run's own
  `outputs/**/subpopulation_distances.csv` row for row, asserted per run in the
  reconciliation cell;
- the pooled row reproduces `outputs/culture/population_adaptability.csv` —
  `e_mean_nemd`, `adaptability_ratio` and `c_center_pop_nemd` to the last digit.

One column deliberately does not match. `combined-metric-inputs.csv` spells its
`center_nEMD` from `summary_metrics.csv`, which pools *respondents*; the
`c_center_nemd` here pools *retained cells*, which is the quantity the centers
plates already draw. For Gemma-4-31B-it base on happiness, FA, that is 0.1189
against 0.1261. Neither is wrong; they are different poolings, and this notebook
uses the cell one throughout so the centre term sits on the same footing as the
other three.

## The groups

Seven families, parsed out of the subpopulation label itself:

| family | levels |
|---|---|
| `country` | Australia, Germany, Mexico, Russia, United States |
| `wave` | 1995-1997, 2005-2006, 2017-2018 |
| `sex` | Female, Male |
| `age` | `<25` through `75+`, seven bands |
| `education` | Low, Middle, High |
| `employment` | Working, Student, Homemaker, Unemployed, Retired |
| `marital_status` | Single, Cohabiting, Married, Divorced or separated, Widowed |

From the **label**, not from the respondents inside the cell. The upstream
assignment is a matching, not a strict partition: the cell
`Mexico 2005 Female <25 High Working Single` holds 23 respondents recorded High,
11 Middle and 3 Low. The label is what defines the cell, so the label is what
defines its group. Seven cells carry an `NA` token — two on age, five on
employment — and sit out that one family while counting in every other.

Dispersion and structure are within-group quantities and are recomputed inside
each group, never averaged down from the pooled figure: a group's `AR` compares
that group's own model spread against that group's own survey spread, and its
`rho` ranks only the pairs both cells of which are in the group. Accuracy and
center restrict the same way. The pooled row is therefore not the mean of its
groups and is not meant to be.

## What lands where

`figures/fidelity/` — the root holds the report over every retained cell of a
run pooled, and one subfolder per family holds the same metrics split by that
family's levels.

| plate | folder | reads |
|---|---|---|
| `fig_fidelity_ranking_<view>_<mode>` | root | PFS per run, ranked |
| `fig_fidelity_components_<view>_<mode>` | root | the three terms and the centre behind each PFS |
| `fig_fidelity_center_<view>_<mode>` | root | PFS against centre alignment, base joined to its finetuned variant |
| `fig_fidelity_shift_<view>_<mode>` | root | change in PFS against change in centre, finetuned variants only |
| `fig_fidelity_cells_<view>_<mode>` | root | the per-cell nEMD spread the accuracy term averages |
| `fig_fidelity_<family>_<view>_<mode>` | `<family>/` | PFS, runs against that family's levels |
| `fig_fidelity_<family>_components_<view>_<mode>` | `<family>/` | the same four terms, one panel each |
| `fig_fidelity_<family>_shift_<view>_<mode>` | `<family>/` | change in PFS against the untuned base |

| encoding | meaning |
|---|---|
| colour + marker | the model |
| fill | hollow = untuned `base`, solid = culture-finetuned |
| bright = high | every score heatmap runs 0 to 1 on `viridis`, so bright is good — the reverse of the error heatmaps elsewhere, which run `viridis_r` |
| green / red | only on the shift heatmaps, centred on zero |
| dashed rule | a **fixed reference, never a fit**: Mixtral archived on the ranking and cell plates, zero change on the shift plates |

Five views per plate: `all`, where PFS is the geometric mean across the four
topics, then happiness, politics, religion and trust alone. Two modes. nEMD
scales with a topic's answer count, so a topic view and the `all` view are not
on the same scale and are not read against each other.

Writes `outputs/culture/population_fidelity_{cells,groups,overall}.csv` and
260 plates under `figures/fidelity/`, `.png` and `.pdf` each, 520 files. Reads
nothing the adaptability and structure notebooks write, so it runs on its own.

A **variant** is one of the conditions a base model is evaluated under: `base`,
the untuned model with nothing attached, or a culture-finetuned variant such as
`german`. The CSV column and the run-directory path segment are both still
spelled `arm`.

`.venv` has no jupyter and a bare `uv run` must not be used here — it destroys
the hand-built sm_120 `llama-cpp-python` wheel. Committed without cell outputs.

In [ ]:
from culture.fidelity import (
    CELLS_TABLE,
    GROUPS_TABLE,
    OVERALL_TABLE,
    build_fidelity,
    cell_summary,
    check_fidelity,
    cross_check_cells,
    overall_fidelity,
)
from culture.fidelity_plates import fidelity_plates
from machine_bias_reproduction.io_utils import write_csv

## Build

In [ ]:
cells, groups = build_fidelity()
check_fidelity(cells, groups)
print(len(groups), "group rows over", cells["subpopulation"].nunique(), "cells")

## Reconciling against the split outputs

Each run wrote its own per-cell distances at analysis time. This asserts that
what the notebook just recomputed is those same numbers, run by run, before any
of it is aggregated or drawn.

In [ ]:
print(cross_check_cells(cells), "runs match their subpopulation_distances.csv")

## The tables

In [ ]:
overall = overall_fidelity(groups)
write_csv(cell_summary(cells), CELLS_TABLE)
write_csv(groups, GROUPS_TABLE)
write_csv(overall, OVERALL_TABLE)
print(len(overall), "rows aggregated across the four topics")

## The plates

In [ ]:
files = fidelity_plates(groups, overall)
print(len(files), "files")

## Reading the plates

Start at the components plate, not the ranking. Nearly every run on these
plates scores high on accuracy, middling on dispersion and near zero on
structure, and the geometric mean is doing what it was built to do: a model
that lands close to each cell while ranking the cells wrongly is not
reproducing the population, and PFS says so. A ranking read without the
components looks like a leaderboard of models; read with them it is a
leaderboard of one term.

The centre plate is where the supervisor's question gets its answer. A run in
the bottom right has relocated its average opinion toward the German pool while
staying flat across cells — the cheap way to look adapted, and the reason
centre is kept out of the score. A run that moved up and right earned both. The
dotted connector runs from a base variant to its own finetuned variant, so its
slope is the trade that finetuning made.

The shift plate is the same trade differentially: right of the vertical rule is
fidelity gained, above the horizontal rule is centre gained, and the top-left
quadrant — centre bought at fidelity's expense — is the one to check first.

In the group folders, use the four topic views, not the `all` view. `all` is
the geometric mean across topics and collapses wherever any single topic scored
zero, so those heatmaps come out mostly dark and say only that the collapse
happened; the topic views next to them carry the actual comparison. Nothing is
hidden by this — it is the same knife-edge compounding one level up — but it is
the first thing to trip over when opening a group folder.

Read across a row before reading down a column. A row
that is bright everywhere except Germany is a model the German finetuning made
worse where it was fitted; a row that is bright only on Germany bought its home
cells with everyone else's. The shift heatmaps make that comparison directly,
and the components grid says which of the four terms moved.

The p-values in `rho_structure_p_value` are descriptive only — pairwise
distances over shared cells are not independent — and nothing here is a
significance claim. A group with fewer than three cells gets no `rho` and
therefore no PFS; none of the seven families has one.

Two rows carry `model_flat`: Terra base on trust, full answers, hands Mexico
and Russia one identical distribution for every cell in the country. Every
pairwise distance is then zero, `rho` is undefined rather than low, and the
table keeps it as blank. The score does not: a model with no within-group
variation reproduces none of the group's structure by construction, so
`score_structure` is 0 there and PFS is 0 with it — which is where `AR = 0`
had already put it. `model_flat` is the flag to filter on if that distinction
matters.